In [63]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm import tqdm
import seaborn as sns
import uproot, wandb, os, logging, json, random
import awkward as ak
# import torch
from sklearn.metrics import roc_curve, roc_auc_score, auc, accuracy_score, log_loss
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from sklearn.model_selection import RandomizedSearchCV, GridSearchCV, train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from scipy.stats import randint, uniform
from matplotlib.ticker import FormatStrFormatter
import plotly.graph_objects as go

# ML model
from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import roc_auc_score, roc_curve, log_loss, accuracy_score
from sklearn.neural_network import MLPClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier


# Set up plot defaults
import matplotlib as mpl
mpl.rcParams['figure.figsize'] = 12.0,8.0  # Roughly 11 cm wde by 8 cm high
mpl.rcParams['font.size'] = 14.0 # Use 14 point font
sns.set(style="whitegrid")

# Check for gpu
# torch.cuda.is_available()

In [57]:
path = '/data/jlai/ntups/csv/BDT_input_basic.csv'
df = pd.read_csv(path)
df.describe()

,metsig,metsigres,met,met_noJVT,dmet,ph_pt,ph_eta,ph_phi,jet_central_eta,jet_central_pt1,...,softerm,jetterm,jetterm_sumet,dphi_met_central_jet,balance,dphi_jj,BDTScore,n_jet_central,weights,label
count,2.374471e+06,2.374471e+06,2.374471e+06,2.374471e+06,2.374471e+06,2.374471e+06,2.374471e+06,2.374471e+06,2.374471e+06,2.374471e+06,...,2.374471e+06,2.374471e+06,2.374471e+06,2.374471e+06,2.374471e+06,2.374471e+06,2.374471e+06,2.374471e+06,2.374471e+06,2.374471e+06
mean,8.085141e+00,3.852044e+04,1.901262e+05,1.892991e+05,-8.271095e+02,9.048486e+05,8.146513e-01,-2.072798e-03,-9.753642e+00,8.822327e+05,...,9.443728e+03,8.356989e+05,1.146178e+06,-8.563278e+00,-7.972595e+00,-2.185869e+02,2.216015e-01,2.443367e+00,9.601491e-02,1.565822e-03
std,7.985699e+00,5.911545e+04,1.405338e+05,1.412619e+05,8.991675e+03,8.889817e+05,5.998204e-01,1.821790e+00,9.823051e+01,8.559987e+05,...,6.070485e+03,8.568927e+05,1.071807e+06,9.835174e+01,9.843597e+01,4.144786e+02,5.388693e-02,1.051786e+00,1.908170e+00,3.953949e-02
min,7.113951e-02,8.526343e+03,1.000000e+05,5.930758e+03,-1.712589e+05,5.000019e+04,1.202736e-06,-3.141592e+00,-9.990000e+02,-9.990000e+02,...,0.000000e+00,0.000000e+00,0.000000e+00,-9.990000e+02,-9.990000e+02,-9.990000e+02,1.000001e-01,0.000000e+00,-9.143369e+01,0.000000e+00
25%,2.618015e+00,1.708219e+04,1.162892e+05,1.155593e+05,0.000000e+00,1.767101e+05,3.292853e-01,-1.580835e+00,-6.314996e-01,1.797812e+05,...,4.888379e+03,1.707173e+05,2.673459e+05,1.135227e-01,9.812462e-01,1.014622e-01,1.856679e-01,2.000000e+00,3.252820e-05,0.000000e+00
50%,4.482013e+00,3.244079e+04,1.447310e+05,1.445859e+05,0.000000e+00,5.704866e+05,6.808950e-01,-8.222443e-03,-1.311714e-02,5.641862e+05,...,8.262512e+03,4.793933e+05,7.757431e+05,4.410775e-01,1.178055e+00,9.073769e-01,2.255408e-01,2.000000e+00,1.769263e-03,0.000000e+00
75%,1.201367e+01,5.355994e+04,2.090528e+05,2.092326e+05,0.000000e+00,1.522563e+06,1.171449e+00,1.575179e+00,5.929185e-01,1.425095e+06,...,1.275781e+04,1.370777e+06,1.820485e+06,2.621905e+00,1.660374e+00,2.254813e+00,2.598951e-01,3.000000e+00,1.991303e-02,0.000000e+00
max,1.899870e+02,8.154622e+07,6.400718e+07,6.400718e+07,1.246841e+05,5.011604e+06,2.518129e+00,3.141578e+00,2.499988e+00,5.262966e+06,...,1.022158e+05,5.019984e+06,1.016629e+07,3.141591e+00,2.090643e+02,3.141592e+00,4.375860e-01,4.000000e+00,5.740618e+02,1.000000e+00


In [13]:
df.columns

Index(['metsig', 'metsigres', 'met', 'met_noJVT', 'dmet', 'ph_pt', 'ph_eta',
       'ph_phi', 'jet_central_eta', 'jet_central_pt1', 'jet_central_pt2',
       'dphi_met_phterm', 'dphi_met_ph', 'dphi_met_jetterm',
       'dphi_phterm_jetterm', 'dphi_ph_centraljet1', 'dphi_ph_jet1',
       'metplusph', 'failJVT_jet_pt1', 'softerm', 'jetterm', 'jetterm_sumet',
       'dphi_met_central_jet', 'balance', 'dphi_jj', 'BDTScore',
       'n_jet_central', 'weights', 'process', 'label'],
      dtype='object')

In [68]:
sampled_dfs = []
for proc in df['process'].unique():
    df_proc = df[df['process'] == proc]
    n = min(1000, len(df_proc))
    sampled_dfs.append(df_proc.sample(n=n, random_state=42))

df_sampled = pd.concat(sampled_dfs)

all_processes = list(df_sampled['process'].unique())
base_process = 'ggHyyd'
extra_processes = [p for p in all_processes if p != base_process]

# Create one trace per process
traces = []
for proc in all_processes:
    trace = go.Scatter(
        x=df_sampled[df_sampled['process'] == proc]['dphi_phterm_jetterm'],
        y=1/df_sampled[df_sampled['process'] == proc]['ph_pt'],
        mode='markers',
        name=proc,
        visible=True if proc == base_process else False,
        hoverinfo='skip',
        marker=dict(size=3, opacity=0.4)
    )
    traces.append(trace)

visibility = {proc: (proc == base_process) for proc in all_processes}

buttons = []

for i, proc in enumerate(all_processes):
    if proc == base_process:
        continue

    def make_toggle_button(proc_name=proc, index=i):
        return dict(
            label=f"{proc_name}",
            method="update",
            args=[{"visible": []}],  # Placeholder, filled below
            execute=True
        )

    buttons.append(make_toggle_button())

for b, proc in zip(buttons, extra_processes):
    def get_visibility(proc_clicked):
        def callback(trace_visibility):
            # Toggle clicked process, always keep base_process visible
            trace_visibility[proc_clicked] = not trace_visibility[proc_clicked]
            trace_visibility[base_process] = True
            return [trace_visibility[p] for p in all_processes]
        return callback

    b["args"][0]["visible"] = get_visibility(proc)(visibility.copy())

def update_button_visibility_states():
    for b, proc in zip(buttons, extra_processes):
        b["args"][0]["visible"] = get_visibility(proc)(visibility.copy())

fig = go.Figure(data=traces)

fig.update_layout(
    updatemenus=[
        dict(
            type="buttons",
            direction="down",
            buttons=buttons,
            showactive=False,
            x=1.05,
            xanchor="left",
            y=0.9,
            yanchor="top"
        )
    ],
    xaxis=dict(title="dphi_phterm_jetterm", range=[0, np.pi]),
    yaxis=dict(title="1/ph_pt", range=[1/300000, 1/50000]),
    title="dphi_phterm_jetterm vs 1/ph_pt",
    width=1300,
    height=1050,
    margin=dict(r=300)  # Give space for buttons on the right
)

fig.show()
fig.write_html("/data/jlai/dark_photon/tmp1.html", include_plotlyjs=False, full_html=False)

In [70]:
import plotly.graph_objects as go

# Data for three example processes
x = [1, 2, 3, 4, 5]
y1 = [10, 14, 12, 17, 13]  # base process (always on)
y2 = [9, 13, 15, 16, 12]
y3 = [8, 12, 11, 18, 14]

# Create traces
trace_base = go.Scatter(x=x, y=y1, mode='markers', name='Process A (base)', marker=dict(size=10))
trace_b = go.Scatter(x=x, y=y2, mode='markers', name='Process B', marker=dict(size=10), visible=False)
trace_c = go.Scatter(x=x, y=y3, mode='markers', name='Process C', marker=dict(size=10), visible=False)

# Add all traces
fig = go.Figure(data=[trace_base, trace_b, trace_c])

# Button-based visibility toggles
fig.update_layout(
    updatemenus=[
        dict(
            type="buttons",
            direction="down",
            buttons=[
                dict(label="Toggle B",
                     method="restyle",
                     args=[{"visible": [True, not trace_b.visible, trace_c.visible]}]),
                dict(label="Toggle C",
                     method="restyle",
                     args=[{"visible": [True, trace_b.visible, not trace_c.visible]}]),
                dict(label="Show All",
                     method="restyle",
                     args=[{"visible": [True, True, True]}]),
                dict(label="Hide All (except A)",
                     method="restyle",
                     args=[{"visible": [True, False, False]}]),
            ],
            x=1.2,
            xanchor="left",
            y=1,
            yanchor="top",
            showactive=False
        )
    ],
    title="Simple Scatter Plot with Toggle Buttons",
    xaxis_title="X Axis",
    yaxis_title="Y Axis",
    height=500,
    width=700,
    margin=dict(r=200)
)

fig.show()
fig.write_html("/data/jlai/dark_photon/tmp1.html", include_plotlyjs=False, full_html=False)


In [62]:
path = '/data/jlai/ntups/csv/BDT_input_basic.csv'
df = pd.read_csv(path)
df.describe()

sampled_dfs = []
for proc in df['process'].unique():
    df_proc = df[df['process'] == proc]
    n = min(80000, len(df_proc))
    sampled_dfs.append(df_proc.sample(n=n, random_state=42))

df_sampled = pd.concat(sampled_dfs)

all_processes = list(df_sampled['process'].unique())
base_process = 'ggHyyd'
extra_processes = [p for p in all_processes if p != base_process]

# Create one trace per process
traces = []
for proc in all_processes:
    trace = go.Scattergl(
        x=df_sampled[df_sampled['process'] == proc]['dphi_phterm_jetterm'],
        y=df_sampled[df_sampled['process'] == proc]['dphi_ph_centraljet1'],
        mode='markers',
        name=proc,
        visible=True if proc == base_process else False,
        marker=dict(size=3, opacity=0.4)
    )
    traces.append(trace)

visibility = {proc: (proc == base_process) for proc in all_processes}

buttons = []

for i, proc in enumerate(all_processes):
    if proc == base_process:
        continue

    def make_toggle_button(proc_name=proc, index=i):
        return dict(
            label=f"{proc_name}",
            method="update",
            args=[{"visible": []}],  # Placeholder, filled below
            execute=True
        )

    buttons.append(make_toggle_button())

for b, proc in zip(buttons, extra_processes):
    def get_visibility(proc_clicked):
        def callback(trace_visibility):
            # Toggle clicked process, always keep base_process visible
            trace_visibility[proc_clicked] = not trace_visibility[proc_clicked]
            trace_visibility[base_process] = True
            return [trace_visibility[p] for p in all_processes]
        return callback

    b["args"][0]["visible"] = get_visibility(proc)(visibility.copy())

def update_button_visibility_states():
    for b, proc in zip(buttons, extra_processes):
        b["args"][0]["visible"] = get_visibility(proc)(visibility.copy())

fig = go.Figure(data=traces)

fig.update_layout(
    updatemenus=[
        dict(
            type="buttons",
            direction="down",
            buttons=buttons,
            showactive=False,
            x=1.05,
            xanchor="left",
            y=0.9,
            yanchor="top"
        )
    ],
    xaxis=dict(title="dphi_phterm_jetterm", range=[0, np.pi]),
    yaxis=dict(title="dphi_ph_centraljet1", range=[0, np.pi]),
    title="dphi_phterm_jetterm vs dphi_ph_centraljet1",
    width=1400,
    height=1000,
    margin=dict(r=300) 
)

fig.show()
fig.write_html("/data/jlai/dark_photon/tmp1.html", include_plotlyjs=False, full_html=False)